# Exploding Kittens — Hand-Test

Manuelles Durchspielen einer generierten Implementierung **ohne** Generation, Judge oder Check-Pipeline.

**Ziel:** Regelwerk (`inputs/game_rules.pdf`) neben dem Code halten und Kernmechaniken selbst verifizieren.

**Ablauf:**
1. `CODE_PATH` und Spielerzahl setzen
2. `new_game()` ausführen
3. Interaktiv mit `play_session()` oder einzeln `show()` / `play(...)`
4. Checkliste unten abhaken; Notizen in `meeting/10.7/MANUAL_TESTS.txt`

Conda: `boardbench` · Kernel nach Modulwechsel neu starten.

In [ ]:
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = Path(".").resolve()
OUTPUT_DIR = REPO_ROOT / "outputs"
RULES_PATH = REPO_ROOT / "inputs" / "game_rules.pdf"

# Welche Implementierung testen? (expl_<backend>_<os|ag>.py)
CODE_PATH = OUTPUT_DIR / "expl_gpt_ag.py"
NUM_PLAYERS = 4
START_PLAYER = 0
VIEW_PLAYER = 0  # information_state / Spielerperspektive

if not CODE_PATH.exists():
    raise FileNotFoundError(f"Missing {CODE_PATH}")
if not RULES_PATH.exists():
    print(f"WARN: rulebook not at {RULES_PATH} — run: python generation/activate_game.py exploding_kittens")

print(f"code={CODE_PATH.name}  players={NUM_PLAYERS}  view=p{VIEW_PLAYER}")

In [ ]:
def load_game_module(code_path: Path):
    module_name = f"manual_test_{code_path.stem}"
    spec = importlib.util.spec_from_file_location(module_name, code_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"could not import {code_path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


def player_label(game, state) -> str:
    cp = game.current_player(state)
    labels = { -1: "TERMINAL", -2: "CHANCE", -3: "SIMULTANEOUS" }
    return labels.get(cp, f"p{cp}")


def show(game, state, *, view_player: int = VIEW_PLAYER, full: bool = True) -> None:
    print("--- status ---")
    print(f"current={player_label(game, state)}  terminal={game.is_terminal(state)}")
    if game.is_terminal(state):
        print("returns:", game.returns(state))
    print()
    print("--- player view ---")
    print(game.information_state(state, view_player))
    if full:
        print()
        print("--- full debug (cheat sheet) ---")
        print(game.render(state))
    actions = game.legal_actions(state)
    print()
    print(f"--- legal actions ({len(actions)}) ---")
    for i, action in enumerate(actions):
        name = game.action_to_name(action)
        print(f"  [{i:3d}] {action}  (name={name!r})")


def play(game, state, action: str):
    """Apply one action; returns new state."""
    legal = game.legal_actions(state)
    if action not in legal:
        # allow passing normalized name if module accepts it
        resolved = game.name_to_action(action)
        if resolved not in legal and action not in legal:
            raise ValueError(f"Illegal action {action!r}; legal={legal[:8]}{'...' if len(legal) > 8 else ''}")
        action = resolved if resolved in legal else action
    return game.apply_action(state, action)


def play_index(game, state, index: int):
    legal = game.legal_actions(state)
    if not (0 <= index < len(legal)):
        raise IndexError(f"index {index} out of range 0..{len(legal) - 1}")
    return play(game, state, legal[index])


def play_session(game, state, *, view_player: int = VIEW_PLAYER):
    """REPL: number = action index, raw string = action, q = quit."""
    state = state
    while not game.is_terminal(state):
        show(game, state, view_player=view_player, full=False)
        raw = input("> ").strip()
        if raw.lower() in {"q", "quit", "exit"}:
            print("stopped (state unchanged in notebook — re-run new_game() to reset)")
            return state
        if raw.isdigit():
            state = play_index(game, state, int(raw))
        else:
            state = play(game, state, raw)
        print()
    show(game, state, view_player=view_player, full=True)
    return state


def verify_setup(game, state) -> None:
    """Quick sanity checks for deck/hand composition (Judge scenario 5.1)."""
    n = state.num_players
    hand_lens = [len(h) for h in state.hands]
    defuses_in_hand = sum(h.count("entschaerfung") + h.count("defuse") for h in state.hands)
    draw = list(state.draw_pile)
    ek_in_deck = sum(1 for c in draw if c in ("exploding_kitten",))
    defuse_in_deck = sum(1 for c in draw if c in ("entschaerfung", "defuse"))
    expected_draw = 51 - 7 * n
    print(f"players={n}")
    print(f"hand sizes={hand_lens} (expect 8 each)")
    print(f"defuses dealt={defuses_in_hand} (expect {n})")
    print(f"draw pile={len(draw)} (expect {expected_draw})")
    print(f"EK in deck={ek_in_deck} (implementations often use n-1={n - 1})")
    print(f"defuse in deck={defuse_in_deck} (2p variant often 2, else 6-n={6 - n})")
    print(f"start legal actions={len(game.legal_actions(state))}")

In [ ]:
module = load_game_module(CODE_PATH)
Game = module.Game
game = Game(num_players=NUM_PLAYERS, start_player=START_PLAYER)
state = game.initial_state()

show(game, state)
print()
verify_setup(game, state)

## Einzelzüge

```python
state = play(game, state, "pass")          # oder play_index(game, state, 0)
show(game, state)
```

Nach Modulwechsel: Kernel neu starten und Setup-Zellen erneut ausführen.

In [ ]:
# Beispiel: ein Zug (anpassen oder Zelle leer lassen)
# state = play(game, state, "pass")
# show(game, state)

## Interaktive Session

Nummer eingeben = Action-Index, sonst Action-String. `q` beendet.

In [ ]:
state = game.initial_state()
state = play_session(game, state)

## Szenario-Checkliste (Regelwerk + Code)

Aus Judge-Reviews abgeleitet — manuell anstreben oder mit `show()` nach jedem Schritt prüfen.

| # | Szenario | Erwartung | OK? | Notiz |
|---|----------|-----------|-----|-------|
| 1 | Setup 2/4/5 Spieler | 8 Karten/Hand, 1 Defuse/Hand, Deckgröße 51−7n | | |
| 2 | Zug: Karten spielen dann ziehen | Phase `turn`, am Ende Draw | | |
| 3 | Angriff | Nächster Spieler 2 Züge | | |
| 4 | Hops unter Angriff | turns_remaining sinkt, kein vorzeitiger Wechsel | | |
| 5 | Nö! / Doch | Parität entscheidet Cancel vs. Resolve | | |
| 6 | Blick in die Zukunft | Top-3 sichtbar, danach Draw | | |
| 7 | Wunsch | Zielspieler gibt Karte | | |
| 8 | 2er/3er/5er-Kombos | Pair steal, Triple named card, Five from discard | | |
| 9 | Defuse + Einfügen | EK zurück ins Deck, Zug endet | | |
| 10 | EK ohne Defuse | Elimination, terminal bei 1 Überlebenden | | |
| 11 | 2-Spieler-Variante | nur 2 Defuse im Nachziehstapel? EK-Anzahl? | | |
| 12 | information_state | fremde Hände + Deckreihenfolge versteckt | | |

Ergebnisse: `meeting/10.7/MANUAL_TESTS.txt`

## Varianten vergleichen (optional)

Nur Startzustand — zeigt z. B. 287 vs. 6 legale Startaktionen (Five-Combo-Enumeration).

In [ ]:
COMPARE_PATHS = [
    OUTPUT_DIR / "expl_gpt_os.py",
    OUTPUT_DIR / "expl_gpt_ag.py",
    OUTPUT_DIR / "expl_claude_ag.py",
]

for path in COMPARE_PATHS:
    if not path.exists():
        print(f"skip {path.name} (missing)")
        continue
    mod = load_game_module(path)
    g = mod.Game(num_players=NUM_PLAYERS)
    s = g.initial_state()
    print(f"{path.name:22s}  legal_start={len(g.legal_actions(s)):4d}  phase={s.phase if hasattr(s, 'phase') else '?'}")